In [ ]:
from pathlib import Path
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

root = Path.cwd()
data_dir = root / "data"
matches_dir = data_dir / "matches"
aggregates_dir = data_dir / "aggregates"

# Load all aggregate CSV files.
aggregates = {
    csv_path.stem: pd.read_csv(csv_path)
    for csv_path in sorted(aggregates_dir.glob("*.csv"))
}

# Load all match files
matches = {}
for match_folder in sorted(matches_dir.iterdir()):
    if not match_folder.is_dir():
        continue

    match_id = match_folder.name
    files = {
        "dynamic_events": None,
        "phases_of_play": None,
        "match_json": None,
        "tracking": None,
    }

    dynamic_events_path = match_folder / f"{match_id}_dynamic_events.csv"
    phases_of_play_path = match_folder / f"{match_id}_phases_of_play.csv"
    match_json_path = match_folder / f"{match_id}_match.json"
    tracking_path = match_folder / f"{match_id}_tracking_extrapolated.jsonl"

    if dynamic_events_path.exists():
        files["dynamic_events"] = pd.read_csv(dynamic_events_path)
    if phases_of_play_path.exists():
        files["phases_of_play"] = pd.read_csv(phases_of_play_path)
    if match_json_path.exists():
        with match_json_path.open("r", encoding="utf-8") as f:
            files["match_json"] = json.load(f)
    if tracking_path.exists():
        files["tracking"] = pd.read_json(tracking_path, lines=True)

    matches[match_id] = files

print(f"Loaded {len(aggregates)} aggregate files:")
for name, df in aggregates.items():
    print(f"  - {name}: {df.shape}")

print(f"\nLoaded {len(matches)} match folders.")
for mid, payload in matches.items():
    dyn = payload["dynamic_events"].shape if payload["dynamic_events"] is not None else None
    pop = payload["phases_of_play"].shape if payload["phases_of_play"] is not None else None
    trk = payload["tracking"].shape if payload["tracking"] is not None else None
    mjs = "yes" if payload["match_json"] is not None else "no"
    print(f"  - {mid}: dynamic={dyn}, phases={pop}, tracking={trk}, match_json={mjs}")

C:\Users\vigus\AppData\Local\Temp\ipykernel_16832\2390769235.py:36: DtypeWarning: Columns (276) have mixed types. Specify dtype option on import or set low_memory=False.
  files["dynamic_events"] = pd.read_csv(dynamic_events_path)
C:\Users\vigus\AppData\Local\Temp\ipykernel_16832\2390769235.py:36: DtypeWarning: Columns (75,77,184,264) have mixed types. Specify dtype option on import or set low_memory=False.
  files["dynamic_events"] = pd.read_csv(dynamic_events_path)


Loaded 3 aggregate files:
  - aus1league_obraggregates_20242025: (407, 130)
  - aus1league_passingaggregates_20242025: (407, 48)
  - aus1league_physicalaggregates_20242025: (406, 65)

Loaded 10 match folders.
  - 1886347: dynamic=(5079, 294), phases=(454, 44), tracking=(59061, 7), match_json=yes
  - 1899585: dynamic=(4713, 294), phases=(460, 44), tracking=(60530, 7), match_json=yes
  - 1925299: dynamic=(5220, 294), phases=(492, 44), tracking=(61301, 7), match_json=yes
  - 1953632: dynamic=None, phases=(431, 44), tracking=(59250, 7), match_json=yes
  - 1996435: dynamic=(5292, 294), phases=(448, 44), tracking=(57621, 7), match_json=yes
  - 2006229: dynamic=(4991, 294), phases=(438, 44), tracking=(59270, 7), match_json=yes
  - 2011166: dynamic=(3966, 294), phases=(429, 44), tracking=(71851, 7), match_json=yes
  - 2013725: dynamic=(4999, 294), phases=(486, 44), tracking=(70251, 7), match_json=yes
  - 2015213: dynamic=(4582, 294), phases=(506, 44), tracking=(72101, 7), match_json=yes
  - 20

# Soccer Match Feature Engineering

103 raw aggregated features per team per match across 10 A-League matches → `features.csv` (20 rows × 105 columns).

In [2]:
from IPython.display import display

print("=== AGGREGATES (head) ===")
for name, df in aggregates.items():
    print(f"\n{name}: {df.shape}")
    display(df.head())

print("\n=== MATCH FILES (head) ===")
for match_id, payload in matches.items():
    print(f"\nMatch {match_id}")

    for table_name in ["dynamic_events", "phases_of_play", "tracking"]:
        table = payload.get(table_name)
        if table is None:
            print(f"  {table_name}: missing")
        else:
            print(f"  {table_name}: {table.shape}")
            display(table.head())

    match_json = payload.get("match_json")
    if match_json is None:
        print("  match_json: missing")
    else:
        print(f"  match_json: type={type(match_json).__name__}")
        if isinstance(match_json, dict):
            preview_keys = list(match_json.keys())[:10]
            print(f"  match_json keys (first 10): {preview_keys}")
        elif isinstance(match_json, list):
            print(f"  match_json length: {len(match_json)}")
            print(f"  match_json first item type: {type(match_json[0]).__name__ if match_json else 'n/a'}")

=== AGGREGATES (head) ===

aus1league_obraggregates_20242025: (407, 130)


,competition_edition_id,competition_id,competition_name,season_id,season_name,player_id,player_name,player_short_name,player_birthdate,team_id,...,behindrun_count_dangerous_received_p30tip,comingshortrun_count_dangerous_received_p30tip,crossreceiverrun_count_dangerous_received_p30tip,droppingoffrun_count_dangerous_received_p30tip,overlaprun_count_dangerous_received_p30tip,pullinghalfspacerun_count_dangerous_received_p30tip,pullingwiderun_count_dangerous_received_p30tip,aheadoftheballrun_count_dangerous_received_p30tip,supportrun_count_dangerous_received_p30tip,underlaprun_count_dangerous_received_p30tip
0,870,61,AUS - A-League,95,2024/2025,211,Adam Taggart,A. Taggart,1993-06-02,871,...,1.62,0.0,1.11,0.0,0.0,0.0,0.0,1.21,0.40,0.0
1,870,61,AUS - A-League,95,2024/2025,218,Adama Traoré,A. Traoré,1990-02-03,868,...,0.00,0.0,0.57,0.0,0.0,0.0,0.0,0.57,0.00,0.0
2,870,61,AUS - A-League,95,2024/2025,2759,Dino Arslanagić,D. Arslanagić,1993-04-24,1804,...,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.16,0.00,0.0
3,870,61,AUS - A-League,95,2024/2025,2858,Douglas Costa de Souza,Douglas Costa,1990-09-14,869,...,0.71,0.0,0.00,0.0,0.0,0.0,0.0,0.35,0.35,0.0
4,870,61,AUS - A-League,95,2024/2025,2858,Douglas Costa de Souza,Douglas Costa,1990-09-14,869,...,0.00,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.00,0.0



aus1league_passingaggregates_20242025: (407, 48)


,competition_edition_id,competition_id,competition_name,season_id,season_name,player_id,player_name,player_short_name,player_birthdate,team_id,...,pass_count_torun_completed_p30tip,pass_pct_torun_completed,pass_avgxpass_torun_attempted,pass_count_torun_shotwithin10s_p30tip,pass_count_torun_goalwithin10s_p30tip,passopportunity_count_dangerous_p30tip,pass_count_dangerous_attempted_p30tip,pass_pct_dangerous_completed,pass_count_dangerous_completed_p30tip,pass_count_difficultpass_attempted_p30tip
0,870,61,AUS - A-League,95,2024/2025,211,Adam Taggart,A. Taggart,1993-06-02,871,...,6.27,70.46,70.66,1.11,0.20,21.12,5.66,57.14,3.23,5.15
1,870,61,AUS - A-League,95,2024/2025,218,Adama Traoré,A. Traoré,1990-02-03,868,...,13.16,88.46,70.43,2.29,0.00,12.58,5.72,60.00,3.43,10.87
2,870,61,AUS - A-League,95,2024/2025,2759,Dino Arslanagić,D. Arslanagić,1993-04-24,1804,...,5.86,90.24,89.49,0.00,0.00,0.95,0.32,50.00,0.16,4.60
3,870,61,AUS - A-League,95,2024/2025,2858,Douglas Costa de Souza,Douglas Costa,1990-09-14,869,...,12.41,79.55,72.08,3.19,0.35,42.55,11.35,62.50,7.09,13.83
4,870,61,AUS - A-League,95,2024/2025,2858,Douglas Costa de Souza,Douglas Costa,1990-09-14,869,...,17.50,85.71,79.65,4.38,0.00,26.25,8.75,100.00,8.75,13.13



aus1league_physicalaggregates_20242025: (406, 65)


...

  phases_of_play: (437, 44)


,index,match_id,frame_start,frame_end,time_start,time_end,minute_start,second_start,duration,period,...,third_end,penalty_area_end,team_in_possession_width_start,team_in_possession_width_end,team_in_possession_length_start,team_in_possession_length_end,team_out_of_possession_width_start,team_out_of_possession_width_end,team_out_of_possession_length_start,team_out_of_possession_length_end
0,0,2017461,2512,2540,00:00.2,00:03.0,0,0,2.8,1,...,defensive_third,False,40.94,42.11,43.49,53.80,34.80,31.53,41.92,54.89
1,1,2017461,2540,2573,00:03.0,00:06.3,0,3,3.3,1,...,attacking_third,False,42.04,35.13,54.19,64.79,31.55,27.82,55.14,57.02
2,2,2017461,2573,2591,00:06.3,00:08.1,0,6,1.8,1,...,middle_third,False,27.69,23.51,56.97,54.32,34.82,32.71,64.84,63.16
3,3,2017461,2591,2672,00:08.1,00:16.2,0,8,8.1,1,...,attacking_third,True,32.68,29.56,63.32,80.36,23.56,22.04,54.03,26.11
4,4,2017461,2845,2901,00:33.5,00:39.1,0,33,5.6,1,...,middle_third,False,NaN,36.53,NaN,44.17,NaN,41.80,NaN,66.08


  tracking: (71451, 7)


,frame,timestamp,period,ball_data,possession,image_corners_projection,player_data
0,0,NaT,NaN,"{'x': None, 'y': None, 'z': None, 'is_detected...","{'player_id': None, 'group': None}","{'x_top_left': None, 'y_top_left': None, 'x_bo...",[]
1,1,NaT,NaN,"{'x': None, 'y': None, 'z': None, 'is_detected...","{'player_id': None, 'group': None}","{'x_top_left': None, 'y_top_left': None, 'x_bo...",[]
2,2,NaT,NaN,"{'x': None, 'y': None, 'z': None, 'is_detected...","{'player_id': None, 'group': None}","{'x_top_left': None, 'y_top_left': None, 'x_bo...",[]
3,3,NaT,NaN,"{'x': None, 'y': None, 'z': None, 'is_detected...","{'player_id': None, 'group': None}","{'x_top_left': None, 'y_top_left': None, 'x_bo...",[]
4,4,NaT,NaN,"{'x': None, 'y': None, 'z': None, 'is_detected...","{'player_id': None, 'group': None}","{'x_top_left': None, 'y_top_left': None, 'x_bo...",[]


  match_json: type=dict
  match_json keys (first 10): ['id', 'home_team_score', 'away_team_score', 'date_time', 'stadium', 'home_team', 'home_team_kit', 'away_team', 'away_team_kit', 'home_team_coach']


## Data Loading

Scans `data/matches/` dynamically — no hardcoded match IDs. Match `1953632` has no `dynamic_events.csv`; its event features are zero-filled.

In [3]:
# ── Schema Verification ──────────────────────────────────────────────────────
# Run this cell once to confirm exact column values used in feature computation.

sample_mid = next(iter(matches))
sample = matches[sample_mid]

# 1. match.json team structure
mj = sample["match_json"]
print("match_json top-level keys:", list(mj.keys()))
print("home_team keys:", list(mj["home_team"].keys()) if "home_team" in mj else "MISSING")
print("home_team_id:", mj["home_team"].get("id"))
print("away_team_id:", mj["away_team"].get("id"))

# 2. dynamic_events column names (all 290+)
de = sample["dynamic_events"]
print("\n--- dynamic_events dtypes for key columns ---")
key_de_cols = [
    "event_type", "event_subtype",
    "third_start", "channel_start", "penalty_area_start",
    "pass_direction", "player_targeted_penalty_area_reception",
    "lead_to_shot", "lead_to_goal", "dangerous",
    "last_line_break", "second_last_line_break", "first_line_break",
    "pressing_chain_index", "pressing_chain_length",
    "pass_distance", "n_opponents_bypassed", "team_id",
]
for col in key_de_cols:
    if col in de.columns:
        print(f"  {col}: dtype={de[col].dtype}, sample={de[col].dropna().unique()[:5].tolist()}")
    else:
        print(f"  {col}: MISSING")

# 3. phases_of_play column names
pop = sample["phases_of_play"]
print("\n--- phases_of_play dtypes for key columns ---")
key_pop_cols = [
    "team_in_possession_id",
    "team_in_possession_phase_type",
    "team_out_of_possession_phase_type",
    "duration",
    "n_player_possessions_in_phase",
    "team_possession_loss_in_phase",
    "team_possession_lead_to_shot",
]
for col in key_pop_cols:
    if col in pop.columns:
        print(f"  {col}: dtype={pop[col].dtype}, sample={pop[col].dropna().unique()[:5].tolist()}")
    else:
        print(f"  {col}: MISSING")

# 4. event_type & event_subtype value counts
print("\n--- event_type value_counts ---")
print(de["event_type"].value_counts().to_string())

if "event_subtype" in de.columns:
    print("\n--- event_subtype value_counts (top 20) ---")
    print(de["event_subtype"].value_counts().head(20).to_string())

# 5. phase type distributions
print("\n--- team_in_possession_phase_type value_counts ---")
print(pop["team_in_possession_phase_type"].value_counts().to_string())

print("\n--- team_out_of_possession_phase_type value_counts ---")
print(pop["team_out_of_possession_phase_type"].value_counts().to_string())

match_json top-level keys: ['id', 'home_team_score', 'away_team_score', 'date_time', 'stadium', 'home_team', 'home_team_kit', 'away_team', 'away_team_kit', 'home_team_coach', 'away_team_coach', 'home_team_playing_time', 'away_team_playing_time', 'competition_edition', 'match_periods', 'competition_round', 'referees', 'players', 'status', 'home_team_side', 'ball', 'pitch_length', 'pitch_width']
home_team keys: ['id', 'name', 'short_name', 'acronym']
home_team_id: 4177
away_team_id: 1805

--- dynamic_events dtypes for key columns ---
  event_type: dtype=object, sample=['player_possession', 'passing_option', 'on_ball_engagement', 'off_ball_run']
  event_subtype: dtype=object, sample=['pressing', 'pulling_wide', 'recovery_press', 'coming_short', 'behind']
  third_start: dtype=object, sample=['middle_third', 'defensive_third', 'attacking_third']
  channel_start: dtype=object, sample=['center', 'half_space_left', 'wide_left', 'half_space_right', 'wide_right']
  penalty_area_start: dtype=bool

## Scheme Verification

Confirmed field names and exact string values before writing feature filters.

In [9]:
# ── Feature Computation ──────────────────────────────────────────────────────
# 103 raw aggregated features per team per match across 14 tactical themes.

DE_ZERO_COLS = [
    # Pressing
    "count_pressing_chain_events", "sum_pressing_chain_lengths",
    "count_pressing_attacking_third", "count_pressing_middle_third", "count_pressing_defensive_third",
    # Off-ball runs — subtype counts
    "count_off_ball_runs_total",
    "count_runs_behind", "count_runs_coming_short", "count_runs_overlap", "count_runs_pulling_wide",
    "count_runs_support", "count_runs_underlap", "count_runs_dropping_off",
    "count_runs_ahead_of_ball", "count_runs_cross_receiver",
    # Off-ball runs — outcomes & physical
    "count_runs_targeted", "count_runs_received", "count_runs_dangerous",
    "count_runs_sprinting", "sum_runs_distance_covered",
    # Line breaking
    "count_last_line_breaks", "count_second_last_line_breaks", "count_first_line_breaks",
    # Spatial territory
    "count_events_attacking_third", "count_events_penalty_area", "count_events_wide_channels",
    # Passing
    "sum_pass_distance", "count_forward_passes", "count_passes_into_penalty_area",
    "count_long_passes", "count_backward_passes", "count_high_passes",
    "count_carries_8m_plus", "count_forward_momentum_events",
    # Threat creation
    "count_dangerous_actions", "count_events_lead_to_shot", "sum_opponents_bypassed",
    "count_possession_danger_events",
    # On-ball engagement outcomes
    "count_direct_disruptions", "count_direct_regains",
    "count_beaten_by_possession", "count_beaten_by_movement", "count_force_backward_events",
    # Combination play
    "count_give_and_go", "count_initiate_give_and_go",
    "count_one_touch", "count_quick_pass", "count_headers",
    "count_consecutive_on_ball_engagements",
    # Passing options volume
    "sum_n_passing_options", "sum_n_passing_options_line_break",
    "sum_n_off_ball_runs_at_event", "sum_n_simultaneous_runs",
    "sum_n_opponents_overtaken",
    # Defensive line pressure
    "sum_delta_defensive_line_gain", "sum_separation_gain",
    "count_events_inside_defensive_shape",
    # Intent & disruption
    "count_push_defensive_line", "count_break_defensive_line",
    "count_intended_run_behind", "count_events_goal_side",
    # Game state context
    "count_events_winning", "count_events_losing", "count_events_drawing",
    "count_events_after_corner", "count_events_after_free_kick",
    "count_events_after_goal_kick", "count_events_after_throw_in",
]

rows = []

for match_id, payload in matches.items():
    mj = payload["match_json"]
    home_team_id = mj["home_team"]["id"]
    away_team_id = mj["away_team"]["id"]

    de  = payload["dynamic_events"]
    pop = payload["phases_of_play"]

    for team_id in [home_team_id, away_team_id]:
        row = {"match_id": int(match_id), "team_id": team_id}

        if de is not None:
            t = de[de["team_id"] == team_id]

            # ── 1. Pressing ───────────────────────────────────────────────
            row["count_pressing_chain_events"] = int(t["pressing_chain_index"].notna().sum())
            row["sum_pressing_chain_lengths"]  = float(t["pressing_chain_length"].sum())
            press_ev = t[t["pressing_chain_index"].notna()]
            row["count_pressing_attacking_third"] = int((press_ev["third_start"] == "attacking_third").sum())
            row["count_pressing_middle_third"]    = int((press_ev["third_start"] == "middle_third").sum())
            row["count_pressing_defensive_third"] = int((press_ev["third_start"] == "defensive_third").sum())

            # ── 2. Off-ball runs — subtypes ───────────────────────────────
            obr = t[t["event_type"] == "off_ball_run"]
            row["count_off_ball_runs_total"]  = len(obr)
            row["count_runs_behind"]          = int((obr["event_subtype"] == "behind").sum())
            row["count_runs_coming_short"]    = int((obr["event_subtype"] == "coming_short").sum())
            row["count_runs_overlap"]         = int((obr["event_subtype"] == "overlap").sum())
            row["count_runs_pulling_wide"]    = int(obr["event_subtype"].isin(["pulling_wide", "pulling_half_space"]).sum())
            row["count_runs_support"]         = int((obr["event_subtype"] == "support").sum())
            row["count_runs_underlap"]        = int((obr["event_subtype"] == "underlap").sum())
            row["count_runs_dropping_off"]    = int((obr["event_subtype"] == "dropping_off").sum())
            row["count_runs_ahead_of_ball"]   = int((obr["event_subtype"] == "run_ahead_of_the_ball").sum())
            row["count_runs_cross_receiver"]  = int((obr["event_subtype"] == "cross_receiver").sum())
            row["count_runs_targeted"]       = int((obr["targeted"] == True).sum())
            row["count_runs_received"]       = int((obr["received"] == True).sum())
            row["count_runs_dangerous"]      = int((obr["dangerous"] == True).sum())
            row["count_runs_sprinting"]      = int((obr["speed_avg_band"] == "sprinting").sum())
            row["sum_runs_distance_covered"] = float(obr["distance_covered"].sum())

            # ── 3. Line breaking ──────────────────────────────────────────
            row["count_last_line_breaks"]        = int((t["last_line_break"]        == True).sum())
            row["count_second_last_line_breaks"] = int((t["second_last_line_break"] == True).sum())
            row["count_first_line_breaks"]       = int((t["first_line_break"]       == True).sum())

            # ── 4. Spatial territory ──────────────────────────────────────
            row["count_events_attacking_third"] = int((t["third_start"] == "attacking_third").sum())
            row["count_events_penalty_area"]    = int((t["penalty_area_start"] == True).sum())
            row["count_events_wide_channels"]   = int(t["channel_start"].isin(["wide_left", "wide_right"]).sum())

            # ── 5. Passing & ball-carrying style ─────────────────────────
            row["sum_pass_distance"]              = float(t["pass_distance"].sum())
            row["count_forward_passes"]           = int((t["pass_direction"] == "forward").sum())
            row["count_backward_passes"]          = int((t["pass_direction"] == "backward").sum())
            row["count_long_passes"]              = int((t["pass_range"] == "long").sum())
            row["count_passes_into_penalty_area"] = int((t["player_targeted_penalty_area_reception"] == True).sum())
            row["count_high_passes"]              = int((t["high_pass"] == True).sum())
            row["count_carries_8m_plus"]          = int(((t["carry"] == True) & (t["distance_covered"] >= 8)).sum())
            row["count_forward_momentum_events"]  = int((t["forward_momentum"] == True).sum())

            # ── 6. Threat creation ────────────────────────────────────────
            row["count_dangerous_actions"]        = int((t["dangerous"]        == True).sum())
            row["count_events_lead_to_shot"]      = int(t["lead_to_shot"].sum())
            row["sum_opponents_bypassed"]         = float(t["n_opponents_bypassed"].sum())
            row["count_possession_danger_events"] = int((t["possession_danger"] == True).sum())

            # ── 7. On-ball engagement outcomes ────────────────────────────
            obe = t[t["event_type"] == "on_ball_engagement"]
            row["count_direct_disruptions"]    = int((obe["end_type"] == "direct_disruption").sum())
            row["count_direct_regains"]        = int((obe["end_type"] == "direct_regain").sum())
            row["count_beaten_by_possession"]  = int((obe["beaten_by_possession"] == True).sum())
            row["count_beaten_by_movement"]    = int((obe["beaten_by_movement"]   == True).sum())
            row["count_force_backward_events"] = int((t["force_backward"]         == True).sum())

            # ── 10. Combination play ──────────────────────────────────────
            row["count_give_and_go"]                     = int((t["give_and_go"]                    == True).sum())
            row["count_initiate_give_and_go"]            = int((t["initiate_give_and_go"]           == True).sum())
            row["count_one_touch"]                       = int((t["one_touch"]                      == True).sum())
            row["count_quick_pass"]                      = int((t["quick_pass"]                     == True).sum())
            row["count_headers"]                         = int((t["is_header"]                      == True).sum())
            row["count_consecutive_on_ball_engagements"] = int((t["consecutive_on_ball_engagements"] == True).sum())

            # ── 11. Passing options volume ────────────────────────────────
            row["sum_n_passing_options"]            = float(t["n_passing_options"].sum())
            row["sum_n_passing_options_line_break"] = float(t["n_passing_options_line_break"].sum())
            row["sum_n_off_ball_runs_at_event"]     = float(t["n_off_ball_runs"].sum())
            row["sum_n_simultaneous_runs"]          = float(t["n_simultaneous_runs"].sum())
            row["sum_n_opponents_overtaken"]        = float(t["n_opponents_overtaken"].sum())

            # ── 12. Defensive line pressure ───────────────────────────────
            row["sum_delta_defensive_line_gain"]       = float(t["delta_to_last_defensive_line_gain"].sum())
            row["sum_separation_gain"]                 = float(t["separation_gain"].sum())
            row["count_events_inside_defensive_shape"] = int((t["inside_defensive_shape_start"] == True).sum())

            # ── 13. Intent & disruption ───────────────────────────────────
            row["count_push_defensive_line"]  = int((t["push_defensive_line"]  == True).sum())
            row["count_break_defensive_line"] = int((t["break_defensive_line"] == True).sum())
            row["count_intended_run_behind"]  = int((t["intended_run_behind"]  == True).sum())
            row["count_events_goal_side"]     = int((t["goal_side_start"]      == True).sum())

            # ── 14. Game state context ────────────────────────────────────
            row["count_events_winning"] = int((t["game_state"] == "winning").sum())
            row["count_events_losing"]  = int((t["game_state"] == "losing").sum())
            row["count_events_drawing"] = int((t["game_state"] == "drawing").sum())
            gi = t["game_interruption_before"].fillna("")
            row["count_events_after_corner"]    = int(gi.str.contains("corner",    na=False).sum())
            row["count_events_after_free_kick"] = int(gi.str.contains("free_kick", na=False).sum())
            row["count_events_after_goal_kick"] = int(gi.str.contains("goal_kick", na=False).sum())
            row["count_events_after_throw_in"]  = int(gi.str.contains("throw_in",  na=False).sum())

        else:
            for col in DE_ZERO_COLS:
                row[col] = 0

        # ── 8. Attacking phase structure (phases_of_play, in-possession) ──
        ip = pop[pop["team_in_possession_id"] == team_id]
        pt = ip["team_in_possession_phase_type"]

        row["count_possession_phases"]          = len(ip)
        row["sum_possession_duration_s"]        = float(ip["duration"].sum())
        row["sum_player_possessions_in_phases"] = float(ip["n_player_possessions_in_phase"].sum())
        row["count_possession_losses"]          = int(ip["team_possession_loss_in_phase"].sum())
        row["count_phases_lead_to_shot"]        = int(ip["team_possession_lead_to_shot"].sum())
        row["count_phases_lead_to_goal"]        = int(ip["team_possession_lead_to_goal"].sum())

        row["count_buildup_phases"]     = int((pt == "build_up").sum())
        row["sum_buildup_duration_s"]   = float(ip.loc[pt == "build_up",   "duration"].sum())
        row["count_create_phases"]      = int((pt == "create").sum())
        row["sum_create_duration_s"]    = float(ip.loc[pt == "create",     "duration"].sum())
        row["count_finish_phases"]      = int((pt == "finish").sum())
        row["sum_finish_duration_s"]    = float(ip.loc[pt == "finish",     "duration"].sum())
        row["count_direct_phases"]      = int((pt == "direct").sum())
        row["count_quick_break_phases"] = int((pt == "quick_break").sum())
        row["count_set_play_phases"]    = int((pt == "set_play").sum())
        row["count_transition_phases"]  = int((pt == "transition").sum())

        row["count_phases_attacking_third"]        = int((ip["third_start"] == "attacking_third").sum())
        row["count_phases_ended_attacking_third"]  = int((ip["third_end"]   == "attacking_third").sum())
        row["count_phases_defensive_to_attacking"] = int(
            ((ip["third_start"] == "defensive_third") & (ip["third_end"] == "attacking_third")).sum()
        )

        row["sum_team_possession_width"]      = float(ip["team_in_possession_width_start"].sum())
        row["sum_team_possession_length"]     = float(ip["team_in_possession_length_start"].sum())
        row["sum_team_possession_width_end"]  = float(ip["team_in_possession_width_end"].sum())
        row["sum_team_possession_length_end"] = float(ip["team_in_possession_length_end"].sum())

        ip_sorted  = ip.sort_values("frame_start").reset_index(drop=True)
        next_phase = ip_sorted["team_in_possession_phase_type"].shift(-1)
        row["count_buildup_to_create_transitions"] = int(
            ((ip_sorted["team_in_possession_phase_type"] == "build_up") & (next_phase == "create")).sum()
        )
        row["count_create_to_finish_transitions"] = int(
            ((ip_sorted["team_in_possession_phase_type"] == "create") & (next_phase == "finish")).sum()
        )

        # ── 9. Defensive phase shape (phases_of_play, out-of-possession) ──
        oop  = pop[pop["team_in_possession_id"] != team_id]
        oopt = oop["team_out_of_possession_phase_type"]

        row["count_high_block_phases"]     = int((oopt == "high_block").sum())
        row["sum_high_block_duration_s"]   = float(oop.loc[oopt == "high_block",   "duration"].sum())
        row["count_medium_block_phases"]   = int((oopt == "medium_block").sum())
        row["sum_medium_block_duration_s"] = float(oop.loc[oopt == "medium_block", "duration"].sum())
        row["count_low_block_phases"]      = int((oopt == "low_block").sum())
        row["sum_low_block_duration_s"]    = float(oop.loc[oopt == "low_block",    "duration"].sum())

        row["sum_oop_width_start"]  = float(oop["team_out_of_possession_width_start"].sum())
        row["sum_oop_width_end"]    = float(oop["team_out_of_possession_width_end"].sum())
        row["sum_oop_length_start"] = float(oop["team_out_of_possession_length_start"].sum())
        row["sum_oop_length_end"]   = float(oop["team_out_of_possession_length_end"].sum())

        rows.append(row)

# Assemble DataFrame with stable column order
features_df = pd.DataFrame(rows)
id_cols   = ["match_id", "team_id"]
feat_cols = sorted(c for c in features_df.columns if c not in id_cols)
features_df = features_df[id_cols + feat_cols]

assert features_df.shape[0] == 20, f"Expected 20 rows, got {features_df.shape[0]}"
assert features_df.groupby("match_id")["team_id"].count().eq(2).all(), "Some match doesn't have exactly 2 teams"

print(f"Shape: {features_df.shape}")
print(f"Matches: {features_df['match_id'].nunique()}  |  Teams per match: 2 ✓")
print(f"Features: {len(feat_cols)}")
print(f"\nNaN counts (should all be 0):")
nan_counts = features_df.isna().sum()
print(nan_counts[nan_counts > 0].to_string() if nan_counts.any() else "  None — all clean.")
features_df

Shape: (20, 105)
Matches: 10  |  Teams per match: 2 ✓
Features: 103

NaN counts (should all be 0):
  None — all clean.


,match_id,team_id,count_backward_passes,count_beaten_by_movement,count_beaten_by_possession,count_break_defensive_line,count_buildup_phases,count_buildup_to_create_transitions,count_carries_8m_plus,count_consecutive_on_ball_engagements,...,sum_pass_distance,sum_player_possessions_in_phases,sum_possession_duration_s,sum_pressing_chain_lengths,sum_runs_distance_covered,sum_separation_gain,sum_team_possession_length,sum_team_possession_length_end,sum_team_possession_width,sum_team_possession_width_end
0,1886347,4177,146,3,6,9,24,16,100,79,...,11114.11,644.0,1786.3,983.0,4464.07,-2316.76,13554.58,14020.14,10286.48,10670.63
1,1886347,1805,114,7,12,4,39,30,66,91,...,8967.18,529.0,1391.7,590.0,3430.76,-2163.56,10027.43,10995.79,8366.12,8977.55
2,1899585,4177,147,4,10,0,16,10,74,74,...,8237.32,570.0,1519.5,578.0,2802.14,-1586.17,11862.68,12708.04,8433.52,9159.25
3,1899585,867,125,6,10,6,33,23,73,112,...,7814.14,531.0,1481.0,300.0,3479.37,-2137.71,12089.32,13107.05,9870.04,10383.40
4,1925299,1802,220,7,8,1,31,24,113,57,...,15230.38,759.0,2144.9,422.0,3048.44,-3038.80,14101.40,14873.00,10616.28,11050.72
5,1925299,871,110,4,11,0,40,25,62,101,...,7586.08,474.0,1405.6,421.0,3044.57,-1612.46,11230.61,12229.25,8139.45,8764.96
6,1953632,870,0,0,0,0,37,22,0,0,...,0.00,403.0,1116.3,0.0,0.00,0.00,8840.08,9678.97,7755.77,8531.09
7,1953632,2380,0,0,0,0,33,26,0,0,...,0.00,763.0,2189.8,0.0,0.00,0.00,12878.60,13505.97,10835.20,11599.20
8,1996435,869,183,5,7,2,40,31,94,99,...,11106.02,641.0,1797.7,999.0,3731.26,-2056.38,11812.11,12652.59,9063.34,9477.93
9,1996435,866,102,10,6,3,42,28,79,143,...,8820.99,563.0,1464.7,743.0,3497.46,-2162.25,11059.48,11997.28,9463.80,9964.33


## Feature Engineering

Every feature is a raw count, sum, or duration — no ratios or percentages. Defending team phases are identified by filtering `phases_of_play` rows where the possessing team is the opponent.

---

#### Theme 1 — Pressing
Events with a non-null `pressing_chain_index` belong to a coordinated pressing sequence.

- **`count_pressing_chain_events`** / **`sum_pressing_chain_lengths`** — volume and depth of organized pressing
- **`count_pressing_attacking_third`** / **`_middle_third`** / **`_defensive_third`** — where on the pitch pressing occurs; separates high-press teams from reactive pressers

---

#### Theme 2 — Off-Ball Movement
`event_type == 'off_ball_run'`, split by `event_subtype`. All 10 run types are captured.

- **`count_off_ball_runs_total`** — overall movement volume without the ball
- **`count_runs_behind`** — runs in behind the last line; vertical attacking intent
- **`count_runs_coming_short`** — runs toward the ball; short-combination style
- **`count_runs_overlap`** / **`count_runs_underlap`** — wide attacking runs from fullbacks
- **`count_runs_pulling_wide`** — runs stretching the defensive shape horizontally
- **`count_runs_support`** — nearby support options for the ball carrier
- **`count_runs_dropping_off`** — attackers checking back to link play
- **`count_runs_ahead_of_ball`** — forward runs staying level with last line
- **`count_runs_cross_receiver`** — runs made to receive a cross
- **`count_runs_targeted`** / **`count_runs_received`** — how many runs were actually used by the passer
- **`count_runs_dangerous`** — runs flagged as genuinely threatening
- **`count_runs_sprinting`** / **`sum_runs_distance_covered`** — physical output of off-ball movement

---

#### Theme 3 — Line Breaking
Boolean columns flagged on any event that beat a defensive line.

- **`count_last_line_breaks`** — ball past the deepest line; directly threatens goal
- **`count_second_last_line_breaks`** — broke into the block
- **`count_first_line_breaks`** — escaped the pressing line

---

#### Theme 4 — Spatial Territory

- **`count_events_attacking_third`** / **`count_events_penalty_area`** — territorial pressure and box threat
- **`count_events_wide_channels`** — wide play usage

---

#### Theme 5 — Passing & Ball Carrying

- **`sum_pass_distance`** — total metres passed; high = direct/long-ball, low = short combinations
- **`count_forward_passes`** / **`count_backward_passes`** — directional balance of the passing game
- **`count_long_passes`** — aerial and long-range passes (`pass_range == 'long'`)
- **`count_high_passes`** — lofted passes (`high_pass == True`)
- **`count_passes_into_penalty_area`** — passes targeting a receiver inside the box
- **`count_carries_8m_plus`** — carries of 8+ metres; identifies ball-carrying progressors
- **`count_forward_momentum_events`** — any action with positive progressive movement (`forward_momentum == True`)

---

#### Theme 6 — Threat Creation

- **`count_dangerous_actions`** — threats flagged by SkillCorner
- **`count_events_lead_to_shot`** — actions in passages that ended in a shot
- **`sum_opponents_bypassed`** — total opponents beaten in open play
- **`count_possession_danger_events`** — moments of genuine danger while in possession (`possession_danger == True`)

---

#### Theme 7 — On-Ball Engagement Outcomes
`event_type == 'on_ball_engagement'` — quality of defensive duels.

- **`count_direct_disruptions`** / **`count_direct_regains`** — presses that immediately disrupted or won the ball
- **`count_beaten_by_possession`** / **`count_beaten_by_movement`** — times a defender was evaded
- **`count_force_backward_events`** — engagements that forced the opponent to play backward

---

#### Theme 8 — Attacking Phase Structure
From `phases_of_play.csv`, in-possession rows.

- **`count_possession_phases`** / **`sum_possession_duration_s`** — frequency and total time in possession
- **`sum_player_possessions_in_phases`** — total individual ball contacts
- **`count_possession_losses`** / **`count_phases_lead_to_shot`** / **`count_phases_lead_to_goal`** — possession outcomes
- **`count_buildup_phases`** / **`sum_buildup_duration_s`** — patient build-from-back sequences
- **`count_create_phases`** / **`sum_create_duration_s`** — creative phases in combination zones
- **`count_finish_phases`** / **`sum_finish_duration_s`** — phases in the finishing zone
- **`count_direct_phases`** / **`count_quick_break_phases`** / **`count_transition_phases`** / **`count_set_play_phases`** — remaining phase types
- **`count_phases_attacking_third`** — phases that started in the final third
- **`count_phases_ended_attacking_third`** — phases that *ended* in the final third (regardless of where they started)
- **`count_phases_defensive_to_attacking`** — field-spanning sequences: started in defensive third, ended in attacking third
- **`sum_team_possession_width`** / **`_length`** — team shape at the *start* of phases
- **`sum_team_possession_width_end`** / **`_length_end`** — team shape at the *end* of phases; compare to start values to detect shape change during possession
- **`count_buildup_to_create_transitions`** / **`count_create_to_finish_transitions`** — sustained attacking sequences flowing through consecutive phase types

---

#### Theme 9 — Defensive Phase Shape
From `phases_of_play.csv`, out-of-possession rows.

- **`count_high_block_phases`** / **`sum_high_block_duration_s`** — high pressing defence
- **`count_medium_block_phases`** / **`sum_medium_block_duration_s`** — mid-block compactness
- **`count_low_block_phases`** / **`sum_low_block_duration_s`** — deep defensive structure
- **`sum_oop_width_start`** / **`sum_oop_width_end`** — how wide the defending team's block is at the start and end of phases
- **`sum_oop_length_start`** / **`sum_oop_length_end`** — how deep (long) the defending block is; smaller = more compact

- **`count_events_after_throw_in`** — events following a throw-in restart

`chaotic`, `defending_direct`, `defending_quick_break`, `defending_set_play`, and `defending_transition` are excluded — reactive states, not deliberate shape choices.- **`count_events_after_goal_kick`** — events following a goal kick; high values = team builds from goalkeeper

- **`count_events_after_free_kick`** — events following a free kick

---- **`count_events_after_corner`** — events that followed a corner restart (`game_interruption_before` contains `corner`)

- **`count_events_winning`** / **`count_events_drawing`** / **`count_events_losing`** — how many events occur in each game-state; high `count_events_losing` with high pressing = aggressive comebacks

#### Theme 10 — Combination Play

Quick, connective passing patterns that reveal a team's short-combination style.Volume of events split by the team's scoreline situation — same count features, but stratified by score context.

#### Theme 14 — Game State Context

- **`count_give_and_go`** — completed wall-pass sequences (`give_and_go == True`)

- **`count_initiate_give_and_go`** — attempts to start a give-and-go (completed or not)---

- **`count_one_touch`** — one-touch passes; speed of ball movement under pressure

- **`count_quick_pass`** — passes released quickly after receiving- **`count_events_goal_side`** — events occurring while the player is goal-side of their marker (`goal_side_start == True`)

- **`count_headers`** — aerial duels and flick-ons (`is_header == True`)- **`count_intended_run_behind`** — runs designed to get behind the last line

- **`count_consecutive_on_ball_engagements`** — same player involved in back-to-back ball actions; progressive ball carrying- **`count_break_defensive_line`** — actions that successfully broke through a line

- **`count_push_defensive_line`** — actions explicitly aimed at pushing the defensive line back

---

Boolean intent flags that SkillCorner codes per event.

#### Theme 11 — Passing Options Created#### Theme 13 — Intent & Disruption

Sums of the number of options *available* at each moment — not passing rates, but the volume of opportunity generated by team movement.

---

- **`sum_n_passing_options`** — total available passing targets accumulated across all events

- **`sum_n_passing_options_line_break`** — line-breaking pass options created (not necessarily taken)- **`count_events_inside_defensive_shape`** — actions that were initiated while already inside the opponent's defensive block

- **`sum_n_off_ball_runs_at_event`** — number of runs present at each event, summed; measures movement richness- **`sum_separation_gain`** — total separation created from marking defenders; positive = pulling away from markers

- **`sum_n_simultaneous_runs`** — simultaneous off-ball runs created; higher = more complex movement patterns- **`sum_delta_defensive_line_gain`** — total metres gained toward/against the last defensive line across all events; positive = pushed them back

- **`sum_n_opponents_overtaken`** — total opponents left behind by runs and carries (can be negative if opponent recovers)

Direct measures of how aggressively the team attacked the opponent's last line.

---#### Theme 12 — Defensive Line Pressure


In [11]:
# ── Output ───────────────────────────────────────────────────────────────────
output_path = Path.cwd() / "features.csv"
features_df.to_csv(output_path, index=False)
print(f"Saved → {output_path}")
print(f"Shape: {features_df.shape}  ({features_df.shape[0]} rows × {features_df.shape[1]} columns)")
print(f"\nColumns:\n{list(features_df.columns)}")

Saved → c:\Projects\Pipelines\Soccer-Feature-Engineering\features.csv
Shape: (20, 105)  (20 rows × 105 columns)

Columns:
['match_id', 'team_id', 'count_backward_passes', 'count_beaten_by_movement', 'count_beaten_by_possession', 'count_break_defensive_line', 'count_buildup_phases', 'count_buildup_to_create_transitions', 'count_carries_8m_plus', 'count_consecutive_on_ball_engagements', 'count_create_phases', 'count_create_to_finish_transitions', 'count_dangerous_actions', 'count_direct_disruptions', 'count_direct_phases', 'count_direct_regains', 'count_events_after_corner', 'count_events_after_free_kick', 'count_events_after_goal_kick', 'count_events_after_throw_in', 'count_events_attacking_third', 'count_events_drawing', 'count_events_goal_side', 'count_events_inside_defensive_shape', 'count_events_lead_to_shot', 'count_events_losing', 'count_events_penalty_area', 'count_events_wide_channels', 'count_events_winning', 'count_finish_phases', 'count_first_line_breaks', 'count_force_backwa

## Results

**20 rows × 105 columns** — 2 teams × 10 matches, no missing values. Match `1953632` had no `dynamic_events.csv`; its event-derived features are zero-filled rather than dropped.

---

### What the features capture

- **Pressing** — volume and location of coordinated pressing chains
- **Off-ball movement** — 10 run subtypes plus physical output (sprinting, distance covered)
- **Line breaking** — how often and how deep a team penetrated defensive lines
- **Spatial territory** — where on the pitch events occurred (thirds, wide channels, penalty area)
- **Passing & carrying style** — distance, direction, range, height; carries of 8m+; forward momentum
- **Threat creation** — dangerous actions, shots generated, opponents bypassed, possession danger moments
- **On-ball engagement outcomes** — disruptions, regains, and times a defender was beaten
- **Combination play** — give-and-gos, one-touch passes, quick passes, headers
- **Passing options created** — total options and line-breaking options available at each moment
- **Defensive line pressure** — metres gained against the last line; separation from markers
- **Intent & disruption** — explicit flags for pushing/breaking the defensive line, runs in behind, goal-side positioning
- **Game state context** — volume of events while winning, drawing, losing; events following each restart type
- **Attacking phase structure** — phase types, durations, shape dimensions, spatial progressions, transitions
- **Defensive phase shape** — block type (high/medium/low), durations, and team block dimensions

---

### Limitations

- **10 matches only** — patterns are illustrative, not statistically robust
- **Match `1953632`** — dynamic event features zero-filled for both teams
- **Absolute counts** — not adjusted for opponent strength, style, or possession share
- **No half-level split** — all features are whole-match totals

---

### Next steps

- Split features by match half
- Roll up to season-level profiles per team
- Cluster teams by feature vectors to identify playing style archetypes
- Test whether structural features improve match outcome prediction beyond box-score inputs

## Output

Saves `features.csv` to the project root — 20 rows (one per team per match), 105 columns (`match_id`, `team_id`, and 103 raw features).